In [89]:
import pandas as pd
import re
import fastf1
cache_dir='fastf1_cache'
fastf1.Cache.enable_cache(cache_dir)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [90]:
season=fastf1.get_event_schedule(2025)
# season.info()
season.head()
# session=fastf1.get_session(2025,'bahrain','r')
# session.load()
# session.results.info()
# session.results.head()
# session_quali=fastf1.get_session(2025,'bahrain','q')
# session_quali.load()
# quali=session_quali.results
# quali.info()
# quali.head()
# session_sprintquali=fastf1.get_session(2025,'china','s')
# session_sprintquali.load()
# session_sprintquali.results.head()


,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,Session2,Session2Date,Session2DateUtc,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
0,0,Bahrain,Sakhir,FORMULA 1 ARAMCO PRE-SEASON TESTING 2025,2025-02-28,Pre-Season Testing,testing,Practice 1,2025-02-26 10:00:00+03:00,2025-02-26 07:00:00,Practice 2,2025-02-27 10:00:00+03:00,2025-02-27 07:00:00,Practice 3,2025-02-28 10:00:00+03:00,2025-02-28 07:00:00,None,NaT,NaT,None,NaT,NaT,True
1,1,Australia,Melbourne,FORMULA 1 LOUIS VUITTON AUSTRALIAN GRAND PRIX ...,2025-03-16,Australian Grand Prix,conventional,Practice 1,2025-03-14 12:30:00+11:00,2025-03-14 01:30:00,Practice 2,2025-03-14 16:00:00+11:00,2025-03-14 05:00:00,Practice 3,2025-03-15 12:30:00+11:00,2025-03-15 01:30:00,Qualifying,2025-03-15 16:00:00+11:00,2025-03-15 05:00:00,Race,2025-03-16 15:00:00+11:00,2025-03-16 04:00:00,True
2,2,China,Shanghai,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2025,2025-03-23,Chinese Grand Prix,sprint_qualifying,Practice 1,2025-03-21 11:30:00+08:00,2025-03-21 03:30:00,Sprint Qualifying,2025-03-21 15:30:00+08:00,2025-03-21 07:30:00,Sprint,2025-03-22 11:00:00+08:00,2025-03-22 03:00:00,Qualifying,2025-03-22 15:00:00+08:00,2025-03-22 07:00:00,Race,2025-03-23 15:00:00+08:00,2025-03-23 07:00:00,True
3,3,Japan,Suzuka,FORMULA 1 LENOVO JAPANESE GRAND PRIX 2025,2025-04-06,Japanese Grand Prix,conventional,Practice 1,2025-04-04 11:30:00+09:00,2025-04-04 02:30:00,Practice 2,2025-04-04 15:00:00+09:00,2025-04-04 06:00:00,Practice 3,2025-04-05 11:30:00+09:00,2025-04-05 02:30:00,Qualifying,2025-04-05 15:00:00+09:00,2025-04-05 06:00:00,Race,2025-04-06 14:00:00+09:00,2025-04-06 05:00:00,True
4,4,Bahrain,Sakhir,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2025,2025-04-13,Bahrain Grand Prix,conventional,Practice 1,2025-04-11 14:30:00+03:00,2025-04-11 11:30:00,Practice 2,2025-04-11 18:00:00+03:00,2025-04-11 15:00:00,Practice 3,2025-04-12 15:30:00+03:00,2025-04-12 12:30:00,Qualifying,2025-04-12 19:00:00+03:00,2025-04-12 16:00:00,Race,2025-04-13 18:00:00+03:00,2025-04-13 15:00:00,True


In [91]:
race_results=[]
quali_results=[]
sprint_race_results = []
sprint_quali_results = []
for index, event in season.iloc[1:].iterrows():
    event_year = event['EventDate'].year
    round_num = event['RoundNumber']
    race_name = event['EventName']
    try:
        #load race session
        race = fastf1.get_session(event_year,round_num,'r')
        race.load()
        print(race.results.columns.tolist())    
        # extracting every race results
    
        
        race_session_results = race.results[['DriverNumber','Abbreviation','FullName','TeamName','GridPosition','ClassifiedPosition','Points','Status']].copy()
        race_session_results['RaceNumber'] = round_num
        race_session_results['RaceName'] = race_name
        race_session_results['RaceDate'] = event['EventDate']
        race_session_results.rename(columns={'ClassifiedPosition':'RacePosition'})
        race_results.append(race_session_results)
        # print(race_results)
        # print(index,event['EventDate'].year)
        
        #load qualifying session
        quali = fastf1.get_session(event_year,round_num,'q')
        quali.load()

        #extracting every qualifying results
        
        quali_session_results = quali.results[['DriverNumber','Abbreviation','FullName','TeamName','Position']].copy()
        quali_session_results['RaceNumber'] = round_num
        quali_session_results['RaceName'] = race_name
        quali_session_results.rename(columns={'Position':'QualiPosition'},inplace=True)
        quali_results.append(quali_session_results)
        print(f"✓ Successfully extracted qualifying {race_name}")

        if event['EventFormat'] == 'sprint_qualifying':
            #loading sprint session
            
            sprint = fastf1.get_session(event_year,round_num,'s')
            sprint.load()
            #extracting every sprint race results
            sprint_session_results = sprint.results[['DriverNumber','Abbreviation','FullName','TeamName','GridPosition','ClassifiedPosition','Points','Status']].copy()
            sprint_session_results['RaceNumber'] = round_num
            sprint_session_results['RaceName'] = race_name
            sprint_session_results.rename(columns={'ClassifiedPosition':'SprintPosition'})
            sprint_race_results.append(sprint_session_results)
            print(f"✓ Successfully extracted sprint {race_name}")

            #loading sprint qualifying session
            sprint_quali = fastf1.get_session(event_year,round_num,'sq')
            sprint_quali.load()
            #extracting every sprint quali results
            sprint_quali_session_results = sprint.results[['DriverNumber','Abbreviation','FullName','TeamName','Position']].copy()
            sprint_quali_session_results['RaceNumber'] = round_num
            sprint_quali_session_results['RaceName'] = race_name
            sprint_quali_session_results.rename(columns={'Position':'SprintQualiPosition'})
            sprint_quali_results.append(sprint_quali_session_results)
            print(f"✓ Successfully extracted sprint quali {race_name}")
    except Exception as e:
        print(f'error extracting {race_name}: {e}')
        continue


races_df = pd.concat(race_results, ignore_index=True)
quali_df = pd.concat(quali_results, ignore_index=True)
sprint_df = pd.concat(sprint_race_results, ignore_index=True)
sprint_quali_df = pd.concat(sprint_quali_results, ignore_index=True)

print(f"\n✓ Extracted {len(races_df)} race results")
print(f"✓ Extracted {len(quali_df)} qualifying results")
print(f"\n✓ Extracted {len(sprint_df)} sprint race results")
print(f"✓ Extracted {len(sprint_quali_df)} sprint qualifying results")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INF

['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '4'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '1', '63', '22', '23', '16', '44', '10', '55', '6', '14', '18', '7', '5', '12', '27', '30', '31', '87']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


✓ Successfully extracted qualifying Australian Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '63', '1', '31', '12', '23', '87', '18', '55', '6', '30', '7', '5', '27', '22', '14', '16', '44', '10']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


...

✓ Successfully extracted sprint quali Qatar Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '63', '16', '14', '5', '31', '6', '22', '87', '55', '30', '12', '18', '44', '23', '27', '10', '43']


✓ Successfully extracted qualifying Abu Dhabi Grand Prix

✓ Extracted 479 race results
✓ Extracted 480 qualifying results

✓ Extracted 120 sprint race results
✓ Extracted 120 sprint qualifying results


In [92]:
def to_snake_case(column):
    column = re.sub(r'(?<!^)(?=[A-Z])', '_', column)
    return column.lower()
quali_df.columns = [to_snake_case(col) for col in quali_df.columns]
races_df.columns = [to_snake_case(col) for col in races_df.columns]

sprint_quali_df.columns = [to_snake_case(col) for col in sprint_quali_df.columns]
sprint_df.columns = [to_snake_case(col) for col in sprint_df.columns]

DATA QUALITY CHECKS

In [97]:
# #how many rows does the 4 dataframes contain
# print('Races:',races_df.shape)
# print('Qualifying:',quali_df.shape)
# print("Sprint Race:", sprint_df.shape)
# print("Sprint Qualifying:", sprint_quali_df.shape)

# #verifying the data looks correct
# races_df.head(20)
# quali_df.head(10)
# sprint_df.head(10)
# # sprint_quali_df.head(10)

# #checking for any missing values
# races_df.isna().sum()
# quali_df.isna().sum()
# quali_null = quali_df[quali_df['QualiPosition'].isna()][['FullName', 'TeamName', 'RaceName','QualiPosition']]
# quali_null
# sao_paulo_quali = fastf1.get_session(2025, 'São Paulo', 'q')
# sao_paulo_quali.load()
# print(sao_paulo_quali.results[['DriverNumber', 'Abbreviation', 'FullName','Position', 'Status']])

# sprint_quali_df.isna().sum()
# sprint_df.isna().sum()

# #Check column names
# quali_df.columns.tolist()
# races_df.columns.tolist()
# sprint_quali_df.columns.tolist()
# sprint_df.columns.tolist()


#Check unique values
# races_df.groupby('abbreviation')['race_name'].count()
# driver_kimi = races_df[races_df['full_name'] == 'Andrea Kimi Antonelli']
# driver_kimi
# races_df['full_name'] = races_df['full_name'].replace('Andrea Kimi Antonelli','Kimi Antonelli')
for df in races_df:
    print(df,races_df[df].nunique())
races_df['classified_position']

driver_number 21
abbreviation 21
full_name 22
team_name 10
grid_position 20
classified_position 23
points 11
status 5
race_number 24
race_name 24
race_date 24


0       1
1       2
2       3
3       4
4       5
       ..
474    16
475    17
476    18
477    19
478    20
Name: classified_position, Length: 479, dtype: object

In [94]:
# every_race = races_df.groupby('RaceName')['RaceNumber'].sum()
# # races_df = races_df.sort_values(by=['RaceNumber','ClassifiedPosition'])
# # races_df
# every_race=every_race.sort_values()
# every_race
spanish_gp= races_df[races_df['race_name']=='Spanish Grand Prix']
# spanish_gp = spanish_gp.groupby('team_name')['full_name'].count()
spanish_gp

,driver_number,abbreviation,full_name,team_name,grid_position,classified_position,points,status,race_number,race_name,race_date
160,81,PIA,Oscar Piastri,McLaren,1.0,1,25.0,Finished,9,Spanish Grand Prix,2025-06-01
161,4,NOR,Lando Norris,McLaren,2.0,2,18.0,Finished,9,Spanish Grand Prix,2025-06-01
162,16,LEC,Charles Leclerc,Ferrari,7.0,3,15.0,Finished,9,Spanish Grand Prix,2025-06-01
163,63,RUS,George Russell,Mercedes,4.0,4,12.0,Finished,9,Spanish Grand Prix,2025-06-01
164,27,HUL,Nico Hulkenberg,Kick Sauber,15.0,5,10.0,Finished,9,Spanish Grand Prix,2025-06-01
165,44,HAM,Lewis Hamilton,Ferrari,5.0,6,8.0,Finished,9,Spanish Grand Prix,2025-06-01
166,6,HAD,Isack Hadjar,Racing Bulls,9.0,7,6.0,Finished,9,Spanish Grand Prix,2025-06-01
167,10,GAS,Pierre Gasly,Alpine,8.0,8,4.0,Finished,9,Spanish Grand Prix,2025-06-01
168,14,ALO,Fernando Alonso,Aston Martin,10.0,9,2.0,Finished,9,Spanish Grand Prix,2025-06-01
169,1,VER,Max Verstappen,Red Bull Racing,3.0,10,1.0,Finished,9,Spanish Grand Prix,2025-06-01


In [95]:
race = fastf1.get_session(2025, 'Spain', 'r')
race.load()

print(len(race.results))
print(
    race.results[
        ['DriverNumber', 'Abbreviation', 'FullName',
         'Position', 'Status', 'Points']
    ]
)

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['81', '4', '16', '63', '27', '44', '6', '10', '14', '1', '30', '5', '22', '55', '43', '31', '87', '12', '23']


19
   DriverNumber Abbreviation           FullName  Position    Status  Points
81           81          PIA      Oscar Piastri       1.0  Finished    25.0
4             4          NOR       Lando Norris       2.0  Finished    18.0
16           16          LEC    Charles Leclerc       3.0  Finished    15.0
63           63          RUS     George Russell       4.0  Finished    12.0
27           27          HUL    Nico Hulkenberg       5.0  Finished    10.0
44           44          HAM     Lewis Hamilton       6.0  Finished     8.0
6             6          HAD       Isack Hadjar       7.0  Finished     6.0
10           10          GAS       Pierre Gasly       8.0  Finished     4.0
14           14          ALO    Fernando Alonso       9.0  Finished     2.0
1             1          VER     Max Verstappen      10.0  Finished     1.0
30           30          LAW        Liam Lawson      11.0  Finished     0.0
5             5          BOR  Gabriel Bortoleto      12.0  Finished     0.0
22       